In [ ]:
# ==========================================================
# 🎬 Hybrid Movie Recommendation System
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

print("Loading datasets...")

# =========================
# 1. LOAD DATA
# =========================

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

# =========================
# 2. PREPROCESS DATA
# =========================

data = pd.merge(ratings, movies, on="movieId")

print("\nMerged dataset shape:", data.shape)

# =========================
# 3. COLLABORATIVE FILTERING
# =========================

print("\nBuilding Collaborative Filtering Model...")

user_movie_matrix = data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
).fillna(0)

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

print("User similarity matrix created!")

# Function: Recommend by User
def recommend_by_user(user_id, top_n=5):

    if user_id not in user_movie_matrix.index:
        return ["User not found"]

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    most_similar_user = similar_users.index[1]

    user_movies = user_movie_matrix.loc[user_id]
    similar_user_movies = user_movie_matrix.loc[most_similar_user]

    recommendations = similar_user_movies[
        (similar_user_movies > 0) & (user_movies == 0)
    ].sort_values(ascending=False)

    return recommendations.head(top_n).index.tolist()


# =========================
# 4. CONTENT-BASED FILTERING
# =========================

print("\nBuilding Content-Based Model...")

movies["genres"] = movies["genres"].str.replace("|", " ")

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

content_similarity = cosine_similarity(tfidf_matrix)

print("Content similarity matrix created!")

# Function: Recommend by Movie
def recommend_by_movie(movie_name, top_n=5):

    if movie_name not in movies["title"].values:
        return ["Movie not found"]

    idx = movies[movies["title"] == movie_name].index[0]

    similarity_scores = list(enumerate(content_similarity[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similar_movies = similarity_scores[1:top_n+1]

    return [movies.iloc[i[0]]["title"] for i in similar_movies]


# =========================
# 5. HYBRID RECOMMENDATION
# =========================

def hybrid_recommend(user_id, movie_name, top_n=5):

    content_recs = set(recommend_by_movie(movie_name, 10))
    collaborative_recs = set(recommend_by_user(user_id, 10))

    hybrid = list(content_recs.union(collaborative_recs))

    if movie_name in hybrid:
        hybrid.remove(movie_name)

    return hybrid[:top_n]


# =========================
# 6. TEST THE SYSTEM
# =========================

print("\n==============================")
print(" SAMPLE RECOMMENDATIONS")
print("==============================")

sample_user = 1
sample_movie = movies["title"].iloc[0]

print("\nSelected User ID:", sample_user)
print("Selected Movie:", sample_movie)

print("\n🎥 Collaborative Recommendations:")
print(recommend_by_user(sample_user, 5))

print("\n🎬 Content-Based Recommendations:")
print(recommend_by_movie(sample_movie, 5))

print("\n🔥 Hybrid Recommendations:")
print(hybrid_recommend(sample_user, sample_movie, 5))


# =========================
# 7. PROJECT SUMMARY
# =========================

print("\n==============================")
print("PROJECT COMPLETED SUCCESSFULLY")
print("==============================")

print("""
This Hybrid Movie Recommendation System combines:

1. Collaborative Filtering (User-User Similarity)
2. Content-Based Filtering (Genre Similarity)
3. Hybrid Recommendation Approach

Technologies Used:
- Python
- Pandas
- NumPy
- Scikit-learn

Dataset:
- MovieLens (movies.csv, ratings.csv)

The system successfully recommends top movies based on
user behavior and movie similarity.
""")